# ML Zoomcamp HW2

In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

cars = pd.read_csv("car_fuel_efficiency_2026.csv")
cars.head()


## Data preparation and EDA

In [ ]:
columns = [
    'engine_displacement',
    'horsepower',
    'vehicle_weight',
    'model_year',
    'fuel_efficiency_mpg'
]

cars = cars.loc[:, columns].copy()
cars.head()


In [ ]:
sns.histplot(data=cars, x='fuel_efficiency_mpg')
plt.show()


## Q1

In [ ]:
cars.isna().sum()


## Q2

In [ ]:
cars['horsepower'].median()


## Prepare and split the dataset

In [ ]:
n_total = len(cars)
n_val = int(0.2 * n_total)
n_test = int(0.2 * n_total)
n_train = n_total - n_val - n_test

np.random.seed(42)
order = np.arange(n_total)
np.random.shuffle(order)

train_idx = order[:n_train]
val_idx = order[n_train:n_train + n_val]
test_idx = order[n_train + n_val:]

train_df = cars.iloc[train_idx].reset_index(drop=True)
val_df = cars.iloc[val_idx].reset_index(drop=True)
test_df = cars.iloc[test_idx].reset_index(drop=True)


## Q3

In [ ]:
features = [
    'engine_displacement',
    'horsepower',
    'vehicle_weight',
    'model_year',
]

def train_linear_regression(X, y):
    X_bias = np.column_stack((np.ones(len(X)), X))
    gram = X_bias.T @ X_bias
    weights = np.linalg.inv(gram) @ X_bias.T @ y
    return weights[0], weights[1:]


def rmse(y_true, y_pred):
    return np.sqrt(np.mean((y_true - y_pred) ** 2))


def prepare_X(frame, fill_value):
    return frame[features].fillna(fill_value).to_numpy()


y_train = train_df['fuel_efficiency_mpg'].to_numpy()
y_val = val_df['fuel_efficiency_mpg'].to_numpy()


In [ ]:
X_train = prepare_X(train_df, fill_value=0)
intercept, coef = train_linear_regression(X_train, y_train)

X_val = prepare_X(val_df, fill_value=0)
y_pred = intercept + X_val @ coef

score_zero = rmse(y_val, y_pred)
round(score_zero, 3)


In [ ]:
hp_mean = train_df['horsepower'].mean()

X_train = prepare_X(train_df, fill_value=hp_mean)
intercept, coef = train_linear_regression(X_train, y_train)

X_val = prepare_X(val_df, fill_value=hp_mean)
y_pred = intercept + X_val @ coef

score_mean = rmse(y_val, y_pred)
round(score_mean, 3)


## Q4

In [ ]:
def train_linear_regression_reg(X, y, r=0.001):
    X_bias = np.column_stack((np.ones(len(X)), X))
    gram = X_bias.T @ X_bias
    regularized = gram + r * np.eye(gram.shape[0])
    weights = np.linalg.inv(regularized) @ X_bias.T @ y
    return weights[0], weights[1:]


X_train = prepare_X(train_df, fill_value=0)
X_val = prepare_X(val_df, fill_value=0)

reg_scores = {}
for r in [0, 0.01, 0.1, 1, 5, 10, 100]:
    intercept, coef = train_linear_regression_reg(X_train, y_train, r=r)
    predictions = intercept + X_val @ coef
    reg_scores[r] = round(rmse(y_val, predictions), 4)

reg_scores


## Q5

In [ ]:
seed_scores = []

for seed in range(10):
    np.random.seed(seed)
    shuffled = np.arange(n_total)
    np.random.shuffle(shuffled)

    train_seed = cars.iloc[shuffled[:n_train]].reset_index(drop=True)
    val_seed = cars.iloc[shuffled[n_train:n_train + n_val]].reset_index(drop=True)

    y_train_seed = train_seed['fuel_efficiency_mpg'].to_numpy()
    y_val_seed = val_seed['fuel_efficiency_mpg'].to_numpy()

    X_train_seed = prepare_X(train_seed, fill_value=0)
    X_val_seed = prepare_X(val_seed, fill_value=0)

    intercept, coef = train_linear_regression(X_train_seed, y_train_seed)
    predictions = intercept + X_val_seed @ coef
    seed_scores.append(rmse(y_val_seed, predictions))

round(np.std(seed_scores), 3)


## Q6

In [ ]:
np.random.seed(9)
shuffled = np.arange(n_total)
np.random.shuffle(shuffled)

train_df = cars.iloc[shuffled[:n_train]].reset_index(drop=True)
val_df = cars.iloc[shuffled[n_train:n_train + n_val]].reset_index(drop=True)
test_df = cars.iloc[shuffled[n_train + n_val:]].reset_index(drop=True)

full_train_df = pd.concat([train_df, val_df], ignore_index=True)

y_full_train = full_train_df['fuel_efficiency_mpg'].to_numpy()
y_test = test_df['fuel_efficiency_mpg'].to_numpy()

X_full_train = prepare_X(full_train_df, fill_value=0)
X_test = prepare_X(test_df, fill_value=0)

intercept, coef = train_linear_regression_reg(X_full_train, y_full_train, r=0.001)
y_pred = intercept + X_test @ coef

rmse(y_test, y_pred)
